# 04 Evaluation And Results

This notebook reads the JSON summaries saved by `03_training_pipeline.ipynb` and turns them into tables, plots, and qualitative comparisons.

Expected inputs under `./results/`:
- `*/metrics_summary.json`
- `*/sample_generations.json`


In [ ]:
from pathlib import Path

repo_dir = Path('/content/DSA5204')
if not repo_dir.exists():
    !git clone https://github.com/NomadZhang/DSA5204.git /content/DSA5204

%cd /content/DSA5204
!pip install -q pandas matplotlib seaborn


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.experiment_utils import list_result_summaries

RESULTS_DIR = Path("./results")
records = list_result_summaries(RESULTS_DIR)
if not records:
    raise FileNotFoundError("No metrics summaries found under ./results. Run 03_training_pipeline.ipynb first.")

results_df = pd.DataFrame(records)
results_df["peak_gpu_memory_gb"] = results_df["peak_gpu_memory_mb"] / 1024
results_df["r"] = pd.to_numeric(results_df["r"], errors="coerce")
results_df["compression_vs_full_ft"] = results_df["total_params"] / results_df["trainable_params"]
results_df = results_df.sort_values(["method", "r"], na_position="first").reset_index(drop=True)

summary_columns = [
    "run_name",
    "status",
    "method",
    "r",
    "alpha",
    "trainable_params",
    "trainable_ratio",
    "compression_vs_full_ft",
    "peak_gpu_memory_gb",
    "step_time_sec",
    "eval_loss",
    "perplexity",
]

display(results_df[summary_columns])


In [ ]:
sns.set_theme(style="whitegrid")
lora_df = results_df[(results_df["method"] == "lora") & (results_df["status"] == "completed")].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(lora_df["r"], lora_df["trainable_params"], marker="o", linewidth=2)
axes[0].set_title("Rank vs Trainable Parameters")
axes[0].set_xlabel("LoRA rank r")
axes[0].set_ylabel("Trainable parameters")

axes[1].plot(lora_df["r"], lora_df["peak_gpu_memory_gb"], marker="o", linewidth=2, color="tab:orange")
axes[1].set_title("Rank vs Peak GPU Memory")
axes[1].set_xlabel("LoRA rank r")
axes[1].set_ylabel("Peak memory (GB)")

axes[2].plot(lora_df["r"], lora_df["perplexity"], marker="o", linewidth=2, color="tab:green")
axes[2].set_title("Rank vs Validation Perplexity")
axes[2].set_xlabel("LoRA rank r")
axes[2].set_ylabel("Perplexity")

plt.tight_layout()
plt.show()


In [ ]:
sample_rows = []
for sample_path in sorted(RESULTS_DIR.glob("*/sample_generations.json")):
    payload = json.loads(sample_path.read_text())
    run_name = sample_path.parent.name
    for index, sample in enumerate(payload.get("samples", []), start=1):
        sample_rows.append(
            {
                "run_name": run_name,
                "sample_id": index,
                "prompt": sample["prompt"],
                "generated_text": sample["generated_text"],
            }
        )

samples_df = pd.DataFrame(sample_rows)
if samples_df.empty:
    print("No qualitative generations found yet.")
else:
    display(samples_df)
